## Лабораторная работа № 1 
## Выполнение разведочного анализа больших данных с использованием фреймворка Apache Spark

### Часть 1

В данной части работы рассмотрены:
* загрузка данных из HDFS;
* базовые преобразования данных;
* загрузка преобразованных данных в таблицу `Apache Airflow`.

Подключим необходимые библиотеки.

In [27]:
import os
from pyspark.sql import SparkSession, DataFrame
from pyspark import SparkConf
from pyspark.sql.functions import (
    regexp_replace,
    regexp_extract_all,
    col,
    lit
)

Сформируем объект конфигурации для `Apache Spark`, указав необходимые параметры.

In [28]:
def create_spark_configuration() -> SparkConf:
    """
    Создает и конфигурирует экземпляр SparkConf для приложения Spark.

    Returns:
        SparkConf: Настроенный экземпляр SparkConf.
    """
    # Получаем имя пользователя
    user_name = os.getenv("USER")
    
    conf = SparkConf()
    conf.setAppName("lab 1 Test")
    conf.setMaster("yarn")
    conf.set("spark.submit.deployMode", "client")
    conf.set("spark.executor.memory", "12g")
    conf.set("spark.executor.cores", "8")
    conf.set("spark.executor.instances", "2")
    conf.set("spark.driver.memory", "4g")
    conf.set("spark.driver.cores", "2")
    conf.set("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.0")
    conf.set("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    conf.set("spark.sql.catalog.spark_catalog", "org.apache.iceberg.spark.SparkCatalog")
    conf.set("spark.sql.catalog.spark_catalog.type", "hadoop")
    conf.set("spark.sql.catalog.spark_catalog.warehouse", f"hdfs:///user/{user_name}/warehouse")
    conf.set("spark.sql.catalog.spark_catalog.io-impl", "org.apache.iceberg.hadoop.HadoopFileIO")

    return conf

Создаём сам объект конфигурации.

In [29]:
conf = create_spark_configuration()

Создаём и выводим на экран сессию `Apache Spark`. В процессе создания сессии происходит подключение к кластеру `Apache Hadoop`, что может занять некоторое время.

In [30]:
spark = SparkSession.builder.config(conf=conf).getOrCreate()
spark

26/09/17 20:39:15 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


Для исследования будем использовать датасет `"Video Game Market Price and Revenue Dataset"`, расположенный на платформе `Kaggle` по адресу https://www.kaggle.com/datasets/amith1707/video-game-market-price-and-revenue-dataset.

Датасет содержит более 5.5 миллионов недельных наблюдений за ~10 000 видеоигр (2019–2024): цены, продажи, вовлечённость, отзывы, платформы, регионы.
Данный датасет загружен в `HDFS` по адресу: `hdfs:///user/user0/datasets/videogames/videogames_processed.csv`
Указываем путь в `HDFS` для файла с данными.

In [31]:
path = "hdfs:///user/user0/datasets/videogames/videogames_processed.csv"

Заполняем датафрейм данными из файла.

In [49]:
df = (spark.read.format("csv")
      .option("header", "true")
      .option("inferSchema", "false")
      .load(path)
)

Выводим фрагмент датафрейма на экран.

In [33]:
df.show()

+---------+---------------+-------------+--------------+----------+------------------+-----------------+------------------+------------+----------+---------------+----------------+------------+---------------+------------------+----------------+------------+----------------+----------------+----------------+----------+----------------+--------------+--------------------+-------------------+---------------+-----------------+---------------+---------------+--------------------+---------------------+------------------+------------------+--------+--------+--------+--------+--------+--------+---------------+-------------------+-------------------+--------------+---------+------+----------+------------+--------------------+----------+------------+------------+-------------+---------------+--------------+---------+---------------+-----------------+-------------------+---------------+--------+---------------+--------------+---------+------------+---------+-----------+---------+------------+---

Очевидно, что в целях сохранения ясности изложения и сокращения расчетного времени имеет смысл рассматривать не все солбцы датасета. Оставим следующие колонки, удалив остальные:

| Название столбца  | Расшифровка |
| ------------- | ------------- |
| game_id                | Идентификатор игры |
| game_name              | Название игры |
| platform               | Платформа (PC, PS5, Xbox, Switch) |
| storefront             | Магазин (Steam, Epic, GOG и т.д.) |
| genre                  | Жанр игры |
| publisher_tier         | Уровень издателя (AAA / AA / Indie) |
| age_rating             | Возрастной рейтинг |
| sale_event_name        | Название события распродажи (или "none") |
| current_price_usd      | Текущая цена в USD |
| original_price_usd     | Цена до скидки в USD |
| discount_pct           | Процент скидки |
| estimated_units_sold   | Оценка проданных копий |
| estimated_revenue_usd  | Оценка выручки в USD |
| concurrent_players     | Одновременные игроки |
| peak_ccu_alltime       | Пиковый онлайн за всё время |
| review_count           | Количество отзывов |
| positive_reviews       | Положительные отзывы |
| negative_reviews       | Отрицательные отзывы |
| steam_rating_pct       | Рейтинг Steam (доля 0–1) |
| user_score             | Оценка пользователей |
| metacritic_score       | Оценка Metacritic |
| wishlist_count         | Число добавлений в вишлист |
| twitch_viewers_proxy   | Прокси-метрика зрителей Twitch |
| youtube_views_proxy    | Прокси-метрика просмотров YouTube |
| reddit_mentions        | Упоминания в Reddit |
| social_hype_index      | Индекс социального хайпа |
| hype_score             | Оценка хайпа |
| virality_score         | Оценка виральности |
| release_year           | Год релиза |
| release_month          | Месяц релиза |
| days_since_release     | Дней с релиза |
| price_na               | Цена в Северной Америке |
| price_eu               | Цена в Европе |
| price_gb               | Цена в Великобритании |
| price_jp               | Цена в Японии |
| price_br               | Цена в Бразилии |
| price_au               | Цена в Австралии |

In [34]:
df = df.select(
    # Идентификаторы
    "game_id", "game_name",
    # Категориальные
    "platform", "storefront", "genre", "publisher_tier",
    "age_rating", "sale_event_name",
    # Числовые — цена и продажи
    "current_price_usd", "original_price_usd", "discount_pct",
    "estimated_units_sold", "estimated_revenue_usd",
    # Числовые — вовлечённость
    "concurrent_players", "peak_ccu_alltime", "review_count",
    "positive_reviews", "negative_reviews", "steam_rating_pct",
    "user_score", "metacritic_score", "wishlist_count",
    # Социальные метрики
    "twitch_viewers_proxy", "youtube_views_proxy", "reddit_mentions",
    "social_hype_index", "hype_score", "virality_score",
    # Временные
    "release_year", "release_month", "days_since_release",
    # Региональные цены
    "price_na", "price_eu", "price_gb", "price_jp", "price_br", "price_au"
)
df.show()

+---------+---------------+---------------+--------------+------+--------------+----------+---------------+-----------------+------------------+------------+--------------------+---------------------+------------------+----------------+------------+----------------+----------------+----------------+----------+----------------+--------------+--------------------+-------------------+---------------+-----------------+----------+--------------+------------+-------------+------------------+--------+--------+--------+--------+--------+--------+
|  game_id|      game_name|       platform|    storefront| genre|publisher_tier|age_rating|sale_event_name|current_price_usd|original_price_usd|discount_pct|estimated_units_sold|estimated_revenue_usd|concurrent_players|peak_ccu_alltime|review_count|positive_reviews|negative_reviews|steam_rating_pct|user_score|metacritic_score|wishlist_count|twitch_viewers_proxy|youtube_views_proxy|reddit_mentions|social_hype_index|hype_score|virality_score|release_yea

Выведем на экран метаданные датасета.

In [35]:
df.printSchema()

root
 |-- game_id: string (nullable = true)
 |-- game_name: string (nullable = true)
 |-- platform: string (nullable = true)
 |-- storefront: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- publisher_tier: string (nullable = true)
 |-- age_rating: string (nullable = true)
 |-- sale_event_name: string (nullable = true)
 |-- current_price_usd: string (nullable = true)
 |-- original_price_usd: string (nullable = true)
 |-- discount_pct: string (nullable = true)
 |-- estimated_units_sold: string (nullable = true)
 |-- estimated_revenue_usd: string (nullable = true)
 |-- concurrent_players: string (nullable = true)
 |-- peak_ccu_alltime: string (nullable = true)
 |-- review_count: string (nullable = true)
 |-- positive_reviews: string (nullable = true)
 |-- negative_reviews: string (nullable = true)
 |-- steam_rating_pct: string (nullable = true)
 |-- user_score: string (nullable = true)
 |-- metacritic_score: string (nullable = true)
 |-- wishlist_count: string (nullable

Видно, что все столбцы датасета содержат строковый тип данных, что не соответствует ожиданиям. Выполним преобразования типов данных некоторых столбцов.

In [36]:
def transform_dataframe(data: DataFrame) -> DataFrame:
    """
    Преобразует столбцы DataFrame в указанные типы данных.
    """
    # --- Числовые (Float) ---
    float_cols = [
        "current_price_usd", "original_price_usd", "discount_pct",
        "estimated_units_sold", "estimated_revenue_usd",
        "concurrent_players", "peak_ccu_alltime",
        "steam_rating_pct", "user_score", "metacritic_score",
        "twitch_viewers_proxy", "youtube_views_proxy", "reddit_mentions",
        "social_hype_index", "hype_score", "virality_score",
        "price_na", "price_eu", "price_gb", "price_jp", "price_br", "price_au"
    ]
    for c in float_cols:
        if c in data.columns:
            data = data.withColumn(c, col(c).cast("Float"))

    # --- Целые (Integer) ---
    int_cols = [
        "review_count", "positive_reviews", "negative_reviews",
        "wishlist_count", "release_year", "release_month",
        "days_since_release"
    ]
    for c in int_cols:
        if c in data.columns:
            data = data.withColumn(c, col(c).cast("Integer"))

    return data

In [37]:
df = transform_dataframe(df)

In [38]:
df.show()

+---------+---------------+---------------+--------------+------+--------------+----------+---------------+-----------------+------------------+------------+--------------------+---------------------+------------------+----------------+------------+----------------+----------------+----------------+----------+----------------+--------------+--------------------+-------------------+---------------+-----------------+----------+--------------+------------+-------------+------------------+--------+--------+--------+--------+--------+--------+
|  game_id|      game_name|       platform|    storefront| genre|publisher_tier|age_rating|sale_event_name|current_price_usd|original_price_usd|discount_pct|estimated_units_sold|estimated_revenue_usd|concurrent_players|peak_ccu_alltime|review_count|positive_reviews|negative_reviews|steam_rating_pct|user_score|metacritic_score|wishlist_count|twitch_viewers_proxy|youtube_views_proxy|reddit_mentions|social_hype_index|hype_score|virality_score|release_yea

In [39]:
df.printSchema()

root
 |-- game_id: string (nullable = true)
 |-- game_name: string (nullable = true)
 |-- platform: string (nullable = true)
 |-- storefront: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- publisher_tier: string (nullable = true)
 |-- age_rating: string (nullable = true)
 |-- sale_event_name: string (nullable = true)
 |-- current_price_usd: float (nullable = true)
 |-- original_price_usd: float (nullable = true)
 |-- discount_pct: float (nullable = true)
 |-- estimated_units_sold: float (nullable = true)
 |-- estimated_revenue_usd: float (nullable = true)
 |-- concurrent_players: float (nullable = true)
 |-- peak_ccu_alltime: float (nullable = true)
 |-- review_count: integer (nullable = true)
 |-- positive_reviews: integer (nullable = true)
 |-- negative_reviews: integer (nullable = true)
 |-- steam_rating_pct: float (nullable = true)
 |-- user_score: float (nullable = true)
 |-- metacritic_score: float (nullable = true)
 |-- wishlist_count: integer (nullable = tru

Видно, что теперь столбцы датафрейма содержат значения корректных типов.

Полученный датафрейм сохраним для дальнейшего использования. Сохранение выполним в таблицу `Apache Iceberg`. 

`Apache Iceberg` — это поддерживающий высокую производительность табличный формат для больших данных.

Сначала создадим базу данных, в которой будет расположена таблица. Во избежание путаницы, **каждую таблицу следует называть с использованием фамилии студента**.

In [40]:
database_name = "krutov_database"

Создадим инструкцию SQL для добавления базы данных в каталог `Apache Spark`.

In [41]:
create_database_sql = f"""
CREATE DATABASE IF NOT EXISTS spark_catalog.{database_name}
"""

In [42]:
spark.sql(create_database_sql)

DataFrame[]

Установим созданную базу данных как текущую.

In [43]:
spark.catalog.setCurrentDatabase(database_name)

И, наконец, записываем преобразованный датафрейм в таблицу `sobd_lab1_table`.

In [46]:
# Сохранение DataFrame в виде таблицы
df.writeTo("sobd_lab1_table").using("iceberg").createOrReplace()
spark.sql("SHOW TABLES IN spark_catalog.krutov_database").show()

# Прочитать таблицу обратно
df_check = spark.table("spark_catalog.krutov_database.sobd_lab1_table")

# Сколько строк
print(f"Строк в таблице: {df_check.count()}")

# Схема
df_check.printSchema()

# Первые строки
df_check.show(5)

26/09/17 20:44:30 WARN HadoopTableOperations: Error reading version hint file hdfs:/user/user0/warehouse/krutov_database/sobd_lab1_table/metadata/version-hint.text
java.io.FileNotFoundException: File does not exist: /user/user0/warehouse/krutov_database/sobd_lab1_table/metadata/version-hint.text
	at org.apache.hadoop.hdfs.server.namenode.INodeFile.valueOf(INodeFile.java:87)
	at org.apache.hadoop.hdfs.server.namenode.INodeFile.valueOf(INodeFile.java:77)
	at org.apache.hadoop.hdfs.server.namenode.FSDirStatAndListingOp.getBlockLocations(FSDirStatAndListingOp.java:159)
	at org.apache.hadoop.hdfs.server.namenode.FSNamesystem.getBlockLocations(FSNamesystem.java:2198)
	at org.apache.hadoop.hdfs.server.namenode.NameNodeRpcServer.getBlockLocations(NameNodeRpcServer.java:795)
	at org.apache.hadoop.hdfs.protocolPB.ClientNamenodeProtocolServerSideTranslatorPB.getBlockLocations(ClientNamenodeProtocolServerSideTranslatorPB.java:468)
	at org.apache.hadoop.hdfs.protocol.proto.ClientNamenodeProtocolPro

+---------------+--------------------+-----------+
|      namespace|           tableName|isTemporary|
+---------------+--------------------+-----------+
|krutov_database|sobd_lab1_process...|      false|
|krutov_database|     sobd_lab1_table|      false|
+---------------+--------------------+-----------+

Строк в таблице: 5584706
root
 |-- game_id: string (nullable = true)
 |-- game_name: string (nullable = true)
 |-- platform: string (nullable = true)
 |-- storefront: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- publisher_tier: string (nullable = true)
 |-- age_rating: string (nullable = true)
 |-- sale_event_name: string (nullable = true)
 |-- current_price_usd: float (nullable = true)
 |-- original_price_usd: float (nullable = true)
 |-- discount_pct: float (nullable = true)
 |-- estimated_units_sold: float (nullable = true)
 |-- estimated_revenue_usd: float (nullable = true)
 |-- concurrent_players: float (nullable = true)
 |-- peak_ccu_alltime: float (nullabl

После успешной записи можно посмотреть, какие таблицы входят в базу данных.

In [47]:
for table in spark.catalog.listTables():
    print(table.name)

sobd_lab1_processed_table
sobd_lab1_table


Обратите внимание, что при необходимости созданные базу данных и таблицу можно удалить следующими командами.

In [48]:
# spark.sql("DROP TABLE spark_catalog.ivanov_database.sobd_lab1_table")
# spark.sql("DROP DATABASE spark_catalog.ivanov_database")

После успешной записи таблицы останавливаем сессию `Apache Spark`.

In [50]:
spark.stop()